In [3]:
# Raw Data Loading
import pandas as pd

df_target = pd.read_csv('./gapminder_internet.csv')
df_target.head()

,country,incomeperperson,internetuserate,urbanrate
0,Afghanistan,NaN,3.654122,24.04
1,Albania,1914.996551,44.989947,46.72
2,Algeria,2231.993335,12.500073,65.22
3,Andorra,21943.339900,81.000000,88.92
4,Angola,1381.004268,9.999954,56.70


In [4]:
# 결측치 처리
df_target = df_target.dropna(axis=0)
df_target.head()

,country,incomeperperson,internetuserate,urbanrate
1,Albania,1914.996551,44.989947,46.72
2,Algeria,2231.993335,12.500073,65.22
3,Andorra,21943.339900,81.000000,88.92
4,Angola,1381.004268,9.999954,56.70
5,Antigua and Barbuda,11894.464070,80.645455,30.46


In [ ]:
# pycounty 라이브러리 사용을 위해 국가명 변경
df_target_change_list = [
    (33, 'Cabo Verde'),
    (35, 'Central African Republic'),
    (41, 'Congo, The Democratic Republic of the'),
    (42, 'Congo'),
    (45, "Côte d'Ivoire"),
    (49, 'Czech Republic'),
    (53, 'Dominican Republic'),
    (83, 'Hong Kong'),
    (100, 'Korea, Republic of'),
    (103, "Lao People's Democratic Republic"),
    (112, 'Macao'),
    (113, 'Republic of North Macedonia'),
    (125, 'Federated States of Micronesia'),
    (183, 'Eswatini'),
    (210, 'Yemen')
]

print(df_target[df_target.index == 35])

for idx, country in df_target_change_list:
    df_target.loc[idx, 'country'] = country

print(df_target[df_target.index == 35])

                 country  incomeperperson  internetuserate  urbanrate
35  Central African Rep.       239.518749         2.300027      38.58
                     country  incomeperperson  internetuserate  urbanrate
35  Central African Republic       239.518749         2.300027      38.58


In [ ]:
# pycountry 라이브러리를 사용하여 국가코드 컬럼 생성
import pycountry

def get_country_code(country):
    try:
        result = pycountry.countries.get(name=country)
        if result:
            return result.alpha_2
        return pycountry.countries.search_fuzzy(country)[0].alpha_2
    except Exception:
        return None

df_target['code'] = df_target['country'].apply(get_country_code)

df_target.head()

,country,incomeperperson,internetuserate,urbanrate,code
1,Albania,1914.996551,44.989947,46.72,AL
2,Algeria,2231.993335,12.500073,65.22,DZ
3,Andorra,21943.339900,81.000000,88.92,AD
4,Angola,1381.004268,9.999954,56.70,AO
5,Antigua and Barbuda,11894.464070,80.645455,30.46,AG


In [7]:
# Wikipedia에서 국가별 인구 데이터 가져오기
url = "https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_population"

tables = pd.read_html(
    url,
    storage_options={"User-Agent": "Mozilla/5.0"}
)

df_population = tables[0]

df_population.head()

,Location,Population,% of world,Date,Source (official or from the United Nations),Notes
0,World,8232000000,100%,13 Jun 2025,UN projection[1][3],NaN
1,India,1417492000,17.3%,1 Jul 2025,Official projection[4],[b]
2,China,1408280000,17.2%,31 Dec 2024,Official estimate[5],[c]
3,United States,340110988,4.1%,1 Jul 2024,Official estimate[6],[d]
4,Indonesia,284438782,3.5%,30 Jun 2025,National annual projection[7],NaN


In [8]:
# 필요한 컬럼만 추출
df_population = df_population[['Location', 'Population']]
df_population.columns = ['country', 'population']
df_population = df_population.drop(index=0)
df_population.head(10)

,country,population
1,India,1417492000
2,China,1408280000
3,United States,340110988
4,Indonesia,284438782
5,Pakistan,241499431
6,Nigeria,223800000
7,Brazil,213421037
8,Bangladesh,169828911
9,Russia,146028325
10,Mexico,130575786


In [ ]:
# df_target과 병합하기 위해 국가명 수정
df_population_change_dict = {
    'Bermuda (United Kingdom)': 'Bermuda',
    'Cape Verde': 'Cabo Verde',
    'DR Congo': 'Congo, The Democratic Republic of the',
    'Ivory Coast': "Côte d'Ivoire",
    'Greenland (Denmark)': 'Greenland',
    'Hong Kong (China)': 'Hong Kong',
    'South Korea': 'Korea, Republic of',
    'Laos': "Lao People's Democratic Republic",
    'Macau (China)': 'Macao',
    'North Macedonia': 'Republic of North Macedonia',
    'Micronesia': 'Federated States of Micronesia',
    'Puerto Rico (United States)': 'Puerto Rico',
    'Slovakia': 'Slovak Republic',
    'East Timor': 'Timor-Leste',
}

df_population['country'] = df_population['country'].replace(df_population_change_dict)
df_population.head(10)

,country,population
1,India,1417492000
2,China,1408280000
3,United States,340110988
4,Indonesia,284438782
5,Pakistan,241499431
6,Nigeria,223800000
7,Brazil,213421037
8,Bangladesh,169828911
9,Russia,146028325
10,Mexico,130575786


In [22]:
# Merge
df = pd.merge(df_target, df_population, how='inner', on='country')
df = df.sort_values(by='code', ascending=True)
df = df.reset_index(drop=True)

df.head(10)

,country,incomeperperson,internetuserate,urbanrate,code,population
0,Andorra,21943.339900,81.000000,88.92,AD,88306
1,United Arab Emirates,21087.394120,77.996781,77.88,AE,10678556
2,Antigua and Barbuda,11894.464070,80.645455,30.46,AG,103603
3,Albania,1914.996551,44.989947,46.72,AL,2363314
4,Armenia,1326.741757,44.001025,63.86,AM,3081100
5,Angola,1381.004268,9.999954,56.70,AO,36170961
6,Argentina,10749.419240,36.000335,92.00,AR,46735004
7,Austria,26692.984110,72.731576,67.16,AT,9200931
8,Australia,25249.986060,75.895654,88.74,AU,27536874
9,Azerbaijan,2344.896916,46.679702,51.92,AZ,10241722


In [23]:
# Raw Data Loading
df_region = pd.read_csv('./continents2.csv')
df_region.head()

,name,alpha-2,alpha-3,country-code,iso_3166-2,region,sub-region,intermediate-region,region-code,sub-region-code,intermediate-region-code
0,Afghanistan,AF,AFG,4,ISO 3166-2:AF,Asia,Southern Asia,NaN,142.0,34.0,NaN
1,Åland Islands,AX,ALA,248,ISO 3166-2:AX,Europe,Northern Europe,NaN,150.0,154.0,NaN
2,Albania,AL,ALB,8,ISO 3166-2:AL,Europe,Southern Europe,NaN,150.0,39.0,NaN
3,Algeria,DZ,DZA,12,ISO 3166-2:DZ,Africa,Northern Africa,NaN,2.0,15.0,NaN
4,American Samoa,AS,ASM,16,ISO 3166-2:AS,Oceania,Polynesia,NaN,9.0,61.0,NaN


In [24]:
# Nambia의 국가 코드가 NA이기 때문에 데이터 불러올때 NaN 값으로 처리됨
# 따라서 국가코드 NA로 변경
# 필요없는 컬럼 삭제 및 필요 컬럼 이름 재설정
df_region_drop_col = [
                    'country-code',
                    'alpha-3',
                    'iso_3166-2',
                    'region-code',
                    'sub-region-code',
                    'intermediate-region-code'
                    ]

df_region_rename_dict = {
                        'alpha-2': 'code',
                        'sub-region': 'sub_region',
                        'intermediate-region': 'intermediate_region',
                        }

df_region.loc[df_region['alpha-2'].isna(), 'alpha-2'] = 'NA'
df_region = df_region.drop(columns=df_region_drop_col)
df_region = df_region.rename(columns=df_region_rename_dict)
df_region.head()

,name,code,region,sub_region,intermediate_region
0,Afghanistan,AF,Asia,Southern Asia,NaN
1,Åland Islands,AX,Europe,Northern Europe,NaN
2,Albania,AL,Europe,Southern Europe,NaN
3,Algeria,DZ,Africa,Northern Africa,NaN
4,American Samoa,AS,Oceania,Polynesia,NaN


In [ ]:
# 컬럼 이름 재설정 및 필요 컬럼 추출
df_rename_dict = {
                'incomeperperson': 'income_per_person',
                'internetuserate': 'internet_use_rate',
                }

new_col_order = [
                'code',
                'country',
                'population',
                'income_per_person',
                'internet_use_rate',
                'urbanrate',
                'region',
                'sub_region',
                'intermediate_region'
                ]

tmp_df = pd.merge(df, df_region, how='inner', on='code').sort_values(by='code', ascending=True)
tmp_df = tmp_df.rename(columns=df_rename_dict)
tmp_df = tmp_df[new_col_order].reset_index(drop=True)
df = tmp_df.copy()
df.head()

,code,country,population,income_per_person,internet_use_rate,urbanrate,region,sub_region,intermediate_region
0,AD,Andorra,88306,21943.339900,81.000000,88.92,Europe,Southern Europe,NaN
1,AE,United Arab Emirates,10678556,21087.394120,77.996781,77.88,Asia,Western Asia,NaN
2,AG,Antigua and Barbuda,103603,11894.464070,80.645455,30.46,Americas,Latin America and the Caribbean,Caribbean
3,AL,Albania,2363314,1914.996551,44.989947,46.72,Europe,Southern Europe,NaN
4,AM,Armenia,3081100,1326.741757,44.001025,63.86,Asia,Western Asia,NaN


In [27]:
# 국가별 인터넷 사용률 및 인당 소득을 대륙별로
# 단순 평균이 아닌 인구수를 고려한 가중 평균 계산
import numpy as np

df_result = df.groupby(['region', 'sub_region']).apply(
    lambda x: pd.Series({
        'weighted_avg_internet': np.average(x['internet_use_rate'], weights=x['population']),
        'weighted_avg_income': np.average(x['income_per_person'], weights=x['population'])
    }),
    include_groups=False
)
df_result.head(10)

weighted_avg_internet  \
region   sub_region                                               
Africa   Northern Africa                              27.533737   
         Sub-Saharan Africa                           12.075633   
Americas Latin America and the Caribbean              33.953552   
         Northern America                             75.019551   
Asia     Central Asia                                 20.417478   
         Eastern Asia                                 39.406379   
         South-eastern Asia                           19.496384   
         Southern Asia                                 8.615410   
         Western Asia                                 29.416232   
Europe   Eastern Europe                               47.943948   

                                          weighted_avg_income  
region   sub_region                                            
Africa   Northern Africa                          2286.767241  
         Sub-Saharan Africa                        688.854607  
Americas Latin America and the Caribbean          4898.061698  
         Northern America                        36188.834373  
Asia     Central Asia                             1286.543092  
         Eastern Asia                             5891.334893  
         South-eastern Asia                       1787.021252  
         Southern Asia                             811.341299  
         Western Asia                             5724.235268  
Europe   Eastern Europe                           3510.003089

In [ ]:
# 중국(CN)과 인도(IN)을 제외한 Asia - Eastern Asia, Southern Asia의
# 인터넷 사용률 및 인당 소득 인구수 고려 가중평균 구하기
df_result = df[
    ((df['code'] != 'CN') & (df['code'] != 'IN')) &
    (df['sub_region'].isin(['Eastern Asia', 'Southern Asia']))
]

df_result = df_result.groupby(['region', 'sub_region']).apply(
    lambda x: pd.Series({
        'weighted_ave_internet': np.average(x['internet_use_rate'], weights=x['population']),
        'weighted_ave_income': np.average(x['income_per_person'], weights=x['population'])
    }),
    include_groups=False
)

df_result

weighted_ave_internet  weighted_ave_income
region sub_region                                               
Asia   Eastern Asia               77.435235         32102.029567
       Southern Asia              11.488742           874.817623